# 1. Setup & Environment
- 민감도 검증용 GIS 및 수치 연산 라이브러리 로드
- 부동소수점 포맷 및 디스플레이 환경 설정

In [1]:
# 1. 라이브러리 로드 및 환경 설정
from pathlib import Path
import json
import math
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: "%.4f" % x)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

# 2. Configuration & Controlled Experiment Definitions
- 입력 데이터 및 OD Matrix 경로 설정
- 연도·시간대 고정 및 3대 속도 조건(congested/normal/freeflow) 통제실험 파라미터 정의

In [2]:
# 2. 경로 및 통제실험 파라미터 정의
BASE_DIR = Path("/mnt/cowork/EV")
OD_DIR = BASE_DIR / "output/g2sfca_sfast_gaussian"
D1_FP = BASE_DIR / "input/processed/서울시_생활인구/집계구_생활인구_원본(OA-14979)/d1_final_2021_2024.csv"
CHARGER_DIR_FASTONLY = BASE_DIR / "input/processed/yearly_snapshots_fastonly"
APT_FP = BASE_DIR / "output/apt_charger_flags/seoul_chargers_2024_apt_v3_final.csv"

DIR_OUTPUT = BASE_DIR / "output"
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

CUTOFF_SEC = 900  # 15분 임계치
YEARS = [2021, 2022, 2023, 2024]
PERIOD_CONFIG = {
    "오전": {"window": (7, 9), "col": "오전_avg"},
    "낮": {"window": (11, 13), "col": "낮_avg"}
}
TRAFFIC_SCENARIOS = ["congested", "normal", "freeflow"]

print(f">> 총 분석 연도: {len(YEARS)}개년 | 시간대: {len(PERIOD_CONFIG)}개 | 속도조건: {len(TRAFFIC_SCENARIOS)}개 (총 {len(YEARS)*len(PERIOD_CONFIG)*len(TRAFFIC_SCENARIOS)}회 실험)")

>> 총 분석 연도: 4개년 | 시간대: 2개 | 속도조건: 3개 (총 24회 실험)


# 3. Decay, Operating Hours & Inequality Functions
- 운영시간 정규식 파서 및 개방 여부 판별 함수
- 벡터화 가우시안 감쇄 함수 및 인구가중 로렌츠 Gini 계수 산출 함수

In [3]:
# 3. 보조 연산 및 지수 산출 함수 정의
TIME_RANGE_RE = re.compile(r"(\d{1,2})[:시](\d{2})?\s*[~-]\s*(\d{1,2})[:시](\d{2})?")

def parse_open_window(text):
    if not text or not str(text).strip() or "24시간" in str(text) or "24시" in str(text):
        return None
    m = TIME_RANGE_RE.search(str(text).strip())
    if not m:
        return None
    h1, _, h2, _ = m.groups()
    start, end = int(h1), int(h2)
    weekday_only = ("평일" in str(text)) or ("주중" in str(text))
    return (start, end, weekday_only)

def is_open(parsed, window_start, window_end, daytype="week"):
    if parsed is None:
        return True
    start, end, weekday_only = parsed
    if weekday_only and daytype == "weekend":
        return False
    if end <= start:
        return True
    return not (end <= window_start or start >= window_end)

def decay_gaussian(tt, d0=CUTOFF_SEC):
    tt = np.asarray(tt, dtype=np.float64)
    w = (np.exp(-0.5 * (tt / d0)**2) - math.exp(-0.5)) / (1.0 - math.exp(-0.5))
    return np.where(tt <= d0, w, 0.0)

def weighted_gini(score: np.ndarray, weight: np.ndarray) -> float:
    df = pd.DataFrame({"score": score, "weight": weight})
    df = df[df["weight"] > 0].dropna()
    if len(df) == 0 or df["weight"].sum() <= 0:
        return np.nan
    
    df = df.sort_values("score").reset_index(drop=True)
    df["mass"] = df["score"] * df["weight"]
    total_w = df["weight"].sum()
    total_mass = df["mass"].sum()
    if total_mass <= 0:
        return np.nan
    
    cum_w = np.concatenate([[0.0], (df["weight"].cumsum() / total_w).values])
    cum_mass = np.concatenate([[0.0], (df["mass"].cumsum() / total_mass).values])
    return float(1.0 - np.sum((cum_w[1:] - cum_w[:-1]) * (cum_mass[1:] + cum_mass[:-1])))

# 4. Vectorized G2SFCA Engine
- OD 행렬 기반 고속 벡터화 가중수요 및 2단계 접근성 점수 산출

In [4]:
# 4. 고속 벡터화 G2SFCA 연산 엔진
def compute_g2sfca(df_od, dict_supply, s_demand, d0=CUTOFF_SEC):
    w = decay_gaussian(df_od["travel_time_sec"].values, d0=d0)
    
    demand_vals = df_od["oa_code"].map(s_demand).fillna(0.0).values
    w_demand = demand_vals * w
    
    df_temp_d = pd.DataFrame({"station_id": df_od["station_id"].values, "w_demand": w_demand})
    d_sum_series = df_temp_d.groupby("station_id")["w_demand"].sum()
    
    supply_series = pd.Series(dict_supply)
    r_j = (supply_series / d_sum_series).replace([np.inf, -np.inf], 0.0).fillna(0.0)
    
    r_vals = df_od["station_id"].map(r_j).fillna(0.0).values
    w_supply = r_vals * w
    
    df_temp_a = pd.DataFrame({"oa_code": df_od["oa_code"].values, "w_supply": w_supply})
    a_i = df_temp_a.groupby("oa_code")["w_supply"].sum()
    
    return a_i.reindex(s_demand.index, fill_value=0.0)

# 5. Data Loaders
- 아파트 충전소(v3), 연도별 생활인구 및 급속충전소 GeoJSON 로더

In [5]:
# 5. 데이터 로더 정의
df_apt = pd.read_csv(APT_FP, dtype={"station_id": str})
apt_set = set(df_apt[df_apt["is_apt_v3"]]["station_id"])
print(f">> 아파트 제외 대상 충전소: {len(apt_set):,}개")

df_pop_all = pd.read_csv(D1_FP, dtype={"집계구코드": str})

def load_demand_series(year: int, col_name: str) -> pd.Series:
    df_y = df_pop_all[df_pop_all["year"] == year]
    return pd.Series(df_y[col_name].astype(float).values, index=df_y["집계구코드"].values)

def load_supply_and_hours(year: int):
    fname = f"metro7_ev_chargers_{year}_fastonly.geojson"
    fp = unicodedata.normalize("NFD", str(CHARGER_DIR_FASTONLY / fname))
    with open(fp, encoding="utf-8") as f:
        data = json.load(f)
    supply, hours = {}, {}
    for feat in data["features"]:
        p = feat["properties"]
        if p.get("city") == "서울특별시":
            sid = str(p["station_id"])
            supply[sid] = float(p.get("fast_count", 0) or 0)
            hours[sid] = parse_open_window(p.get("openinghour", ""))
    return supply, hours

>> 아파트 제외 대상 충전소: 5,634개


# 6. Controlled Experiment Batch Pipeline
- 연도·시간대 고정 상태에서 교통 시나리오별 Reach(도달 충전소 수) 및 Gini 계수 일괄 산출

In [6]:
# 6. 민감도 통제실험 배치 실행
print("=" * 80)
print("RUNNING: TRAFFIC SCENARIO SENSITIVITY & CATCHMENT COMPRESSION PIPELINE")
print("=" * 80)

results = []
daytype = "week"  # 통제실험을 위해 평일 기준 3개 속도 시나리오 비교

for year in YEARS:
    raw_supply, hours_dict = load_supply_and_hours(year)
    
    for period, cfg in PERIOD_CONFIG.items():
        w_start, w_end = cfg["window"]
        s_demand = load_demand_series(year, cfg["col"])
        
        # 유효 공급량 산출
        effective_supply = {}
        for sid, count in raw_supply.items():
            if sid in apt_set or not is_open(hours_dict.get(sid), w_start, w_end, daytype):
                effective_supply[sid] = 0.0
            else:
                effective_supply[sid] = count
                
        for scenario in TRAFFIC_SCENARIOS:
            tag = f"{year}_{daytype}_{period}_{scenario}"
            fp_od = OD_DIR / f"od_{tag}.csv"
            
            if not fp_od.exists():
                print(f"  [!] OD 파일 누락: {fp_od.name}")
                continue
                
            df_od = pd.read_csv(fp_od, dtype={"station_id": str, "oa_code": str})
            
            # G2SFCA 및 지니계수 산출
            s_score = compute_g2sfca(df_od, effective_supply, s_demand, d0=CUTOFF_SEC)
            gini_val = weighted_gini(s_score.values, s_demand.values)
            
            # 집계구당 평균 도달 가능 충전소 수 (Reach)
            reach_val = len(df_od) / df_od["oa_code"].nunique()
            
            print(f"  [>] {year} {period} {scenario:<10} | Reach: {reach_val:6.1f} EA/OA | Gini: {gini_val:.4f}")
            results.append({
                "year": year,
                "period": period,
                "scenario": scenario,
                "reach": reach_val,
                "gini": gini_val
            })

df_sensitivity = pd.DataFrame(results)

RUNNING: TRAFFIC SCENARIO SENSITIVITY & CATCHMENT COMPRESSION PIPELINE
  [>] 2021 오전 congested  | Reach:   55.7 EA/OA | Gini: 0.2743
  [>] 2021 오전 normal     | Reach:  105.0 EA/OA | Gini: 0.1946
  [>] 2021 오전 freeflow   | Reach:  304.5 EA/OA | Gini: 0.1318
  [>] 2021 낮 congested  | Reach:   22.6 EA/OA | Gini: 0.3384
  [>] 2021 낮 normal     | Reach:   47.2 EA/OA | Gini: 0.2590
  [>] 2021 낮 freeflow   | Reach:  223.8 EA/OA | Gini: 0.1304
  [>] 2022 오전 congested  | Reach:   80.4 EA/OA | Gini: 0.2438
  [>] 2022 오전 normal     | Reach:  145.0 EA/OA | Gini: 0.1780
  [>] 2022 오전 freeflow   | Reach:  416.8 EA/OA | Gini: 0.1250
  [>] 2022 낮 congested  | Reach:   33.0 EA/OA | Gini: 0.3072
  [>] 2022 낮 normal     | Reach:   67.4 EA/OA | Gini: 0.2303
  [>] 2022 낮 freeflow   | Reach:  309.0 EA/OA | Gini: 0.1051
  [>] 2023 오전 congested  | Reach:   88.7 EA/OA | Gini: 0.2838
  [>] 2023 오전 normal     | Reach:  180.7 EA/OA | Gini: 0.1942
  [>] 2023 오전 freeflow   | Reach:  597.1 EA/OA | Gini: 0.1208
  [>]

# 7. Summary Pivot Tables, Correlation Analysis & Export
- 속도 조건별 Gini 계수 다중 인덱스 피벗 테이블 생성
- Reach vs Gini 상관계수(Pearson & Spearman) 분석 및 결과 파일(`_mw.csv`) 저장

In [7]:
# 7. 피벗 요약표, 상관계수 산출 및 결과 CSV 저장
print("\n" + "=" * 80)
print("             교통 시나리오별 Gini 계수 변화 피벗 요약표 (평일 기준)")
print("=" * 80)
piv_sens = df_sensitivity.pivot_table(index=["year", "period"], columns="scenario", values="gini")[TRAFFIC_SCENARIOS]
display(piv_sens)

# 상관계수 산출
pearson_r = df_sensitivity["reach"].corr(df_sensitivity["gini"], method="pearson")
spearman_r = df_sensitivity["reach"].corr(df_sensitivity["gini"], method="spearman")

print("\n=== [상관관계 분석 (Reach vs Gini)] ===")
print(f"- Pearson correlation  (r): {pearson_r:7.4f}")
print(f"- Spearman correlation (r): {spearman_r:7.4f}")

# 최종 CSV 저장 (_mw)
out_fp = DIR_OUTPUT / "freeflow_sensitivity_check_mw.csv"
df_sensitivity.to_csv(out_fp, index=False, encoding="utf-8-sig")
print(f"\n>> 민감도 검증 결과 저장 완료: {out_fp}")


             교통 시나리오별 Gini 계수 변화 피벗 요약표 (평일 기준)


scenario     congested  normal  freeflow
year period                             
2021 낮          0.3384  0.2590    0.1304
     오전         0.2743  0.1946    0.1318
2022 낮          0.3072  0.2303    0.1051
     오전         0.2438  0.1780    0.1250
2023 낮          0.3648  0.2895    0.1323
     오전         0.2838  0.1942    0.1208
2024 낮          0.3361  0.2693    0.1447
     오전         0.2613  0.1885    0.1233


=== [상관관계 분석 (Reach vs Gini)] ===
- Pearson correlation  (r): -0.7800
- Spearman correlation (r): -0.8852

>> 민감도 검증 결과 저장 완료: /mnt/cowork/EV/output/freeflow_sensitivity_check_mw.csv


# 8. Result Validation (Comparison with Original Outputs)
- 원본 민감도 검증 파일(`freeflow_sensitivity_check.csv`)과 신규 산출물(`_mw.csv`) 간 수치 오차 전수 대조

In [8]:
# 8. 원본 산출물 vs 신규 산출물(_mw) 정밀 오차 검증
fp_orig = DIR_OUTPUT / "freeflow_sensitivity_check.csv"
fp_new = DIR_OUTPUT / "freeflow_sensitivity_check_mw.csv"

if not fp_orig.exists():
    print(f"[!] 비교할 원본 파일이 존재하지 않습니다: {fp_orig}")
elif not fp_new.exists():
    print(f"[!] 신규 산출물 파일이 생성되지 않았습니다: {fp_new}")
else:
    df_orig = pd.read_csv(fp_orig)
    df_new = pd.read_csv(fp_new)
    
    comp = df_orig.merge(df_new, on=["year", "period", "scenario"], suffixes=("_orig", "_new"))
    
    comp["reach_diff"] = (comp["reach_orig"] - comp["reach_new"]).abs()
    comp["gini_diff"] = (comp["gini_orig"] - comp["gini_new"]).abs()
    
    comp["Status"] = np.where(
        (comp["reach_diff"] < 1e-4) & (comp["gini_diff"] < 1e-5),
        "완전 일치 (정상)",
        "오차 발생 확인 필요"
    )
    
    display_cols = [
        "year", "period", "scenario", "Status",
        "reach_orig", "reach_new", "reach_diff",
        "gini_orig", "gini_new", "gini_diff"
    ]
    
    print("=" * 95)
    print("           민감도 검증 산출물 원본 vs 리팩토링 코드 수치 대조 요약표")
    print("=" * 95)
    display(comp[display_cols])
    
    print("\n=== [정밀 오차 요약] ===")
    print(f"- Reach 최대 오차: {comp['reach_diff'].max():.10e}")
    print(f"- Gini  최대 오차: {comp['gini_diff'].max():.10e}")
    
    if comp["reach_diff"].max() < 1e-4 and comp["gini_diff"].max() < 1e-5:
        print(">> [판정] 모든 통제실험 시나리오의 수치가 원본과 수학적으로 완벽히 일치합니다.")

           민감도 검증 산출물 원본 vs 리팩토링 코드 수치 대조 요약표


,year,period,scenario,Status,reach_orig,reach_new,reach_diff,gini_orig,gini_new,gini_diff
0,2021,오전,congested,완전 일치 (정상),55.7221,55.7221,0.0000,0.2743,0.2743,0.0000
1,2021,오전,normal,완전 일치 (정상),105.0255,105.0255,0.0000,0.1946,0.1946,0.0000
2,2021,오전,freeflow,완전 일치 (정상),304.4802,304.4802,0.0000,0.1318,0.1318,0.0000
3,2021,낮,congested,완전 일치 (정상),22.6132,22.6132,0.0000,0.3384,0.3384,0.0000
4,2021,낮,normal,완전 일치 (정상),47.2470,47.2470,0.0000,0.2590,0.2590,0.0000
5,2021,낮,freeflow,완전 일치 (정상),223.8411,223.8411,0.0000,0.1304,0.1304,0.0000
6,2022,오전,congested,완전 일치 (정상),80.4495,80.4495,0.0000,0.2438,0.2438,0.0000
7,2022,오전,normal,완전 일치 (정상),145.0478,145.0478,0.0000,0.1780,0.1780,0.0000
8,2022,오전,freeflow,완전 일치 (정상),416.8463,416.8463,0.0000,0.1250,0.1250,0.0000
9,2022,낮,congested,완전 일치 (정상),33.0079,33.0079,0.0000,0.3072,0.3072,0.0000



=== [정밀 오차 요약] ===
- Reach 최대 오차: 0.0000000000e+00
- Gini  최대 오차: 6.9388939039e-16
>> [판정] 모든 통제실험 시나리오의 수치가 원본과 수학적으로 완벽히 일치합니다.
